# **Importing Libraries**

In [ ]:
# Standard Library Imports
import os            # OS-level operations (paths, directory handling)
import math          # Mathematical functions
import shutil        # High-level file operations (copying, moving, deleting)

# Data Handling & Processing
import numpy as np               # Numerical computations and array operations
import pandas as pd              # Data manipulation and analysis
import h5py                      # Handling HDF5 file formats

# Progress Visualization
from tqdm import tqdm            # Progress bars for loops

# Plotting & Visualization
import matplotlib.pyplot as plt  # Plotting and visualization tools


# PyTorch Machine Learning Stack
import torch                     # Core PyTorch library
import torch.nn as nn            # Neural network layers and utilities
import torch.optim as optim      # Optimization algorithms (SGD, Adam, etc.)
from torch.utils.data import (   # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)

# **Function to load h5 dataset**

In [ ]:
def load_h5_dataset(file_path):
    """
    Load dataset, labels, and subject IDs from an HDF5 (.h5) file.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing 'data', 'label', and 'sub_id' datasets.

    Returns
    -------
    data : np.ndarray
        The input data stored in the HDF5 file.
    label : np.ndarray
        Corresponding labels for each data sample.
    subjects : np.ndarray
        Subject IDs associated with each data sample.
    """

    # Open the .h5 file in read-only mode to ensure safe, non-destructive access
    with h5py.File(file_path, 'r') as f:
        # Load arrays stored in the HDF5 datasets
        data = np.array(f['data'])       # Main feature/data array
        label = np.array(f['label'])     # Labels or targets for each sample
        subjects = np.array(f['sub_id']) # Subject identifier for each data entry

    # Return loaded components as NumPy arrays
    return data, label, subjects


# **Define pytorch class for data**

In [ ]:
class CustomDatasets(Dataset):
    """
    A PyTorch Dataset wrapper for handling data, labels, and subject IDs.

    This class allows data to be easily fed into a DataLoader for batching,
    shuffling, and parallel loading during training or evaluation.
    """

    def __init__(self, data, labels, subjects):
        """
        Initialize the dataset.

        Parameters
        ----------
        data : array-like
            Input feature data, typically a NumPy array.
        labels : array-like
            Integer class labels for each sample.
        subjects : array-like
            Subject IDs corresponding to each sample (useful for subject-level splits).
        """
        self.data = data
        self.labels = labels
        self.subjects = subjects

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, index):
        """
        Retrieve a single sample by index.

        Returns
        -------
        x : torch.Tensor
            Feature tensor for the given index.
        y : torch.Tensor
            Label tensor for the given index.
        s : torch.Tensor
            Subject ID tensor for the given index.
        """
        # Convert selected sample, label, and subject ID into PyTorch tensors
        x = torch.tensor(self.data[index], dtype=torch.float32)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        s = torch.tensor(self.subjects[index], dtype=torch.long)

        return x, y, s


# **Model architecture**

In [ ]:
# Gradient Reversal Layer (GRL)
# Used in Domain-Adversarial Neural Networks (DANN)
# Reverses gradient during backprop to encourage domain invariance
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        """
        Forward pass: acts as identity function.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
        lambd : float
            Reversal strength; used in the backward pass.
        """
        ctx.lambd = lambd                 # Store λ for backward pass
        return x.view_as(x)               # Identity operation

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: multiply gradient by -λ.
        This forces the feature extractor to learn domain-invariant features.
        """
        return grad_output.neg() * ctx.lambd, None


def grad_reverse(x, lambd):
    """Convenience wrapper for applying the GRL."""
    return GradReverse.apply(x, lambd)


# Token Embedding Module
# Converts raw EEG input into token embeddings suitable for LSTM
class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension for each time step.
        """
        super(TokenEmbedding, self).__init__()

        # First embedding layer:
        # 1 → (d_model*4) channels using temporal convolution
        self.embed_layer = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=d_model * 4,
                kernel_size=(1, 8),
                padding='same'
            ),
            nn.BatchNorm2d(d_model * 4),
            nn.GELU()
        )

        # Second embedding layer:
        # (d_model*4) → d_model channels using spatial convolution across channels
        self.embed_layer2 = nn.Sequential(
            nn.Conv2d(
                in_channels=d_model * 4,
                out_channels=d_model,
                kernel_size=(c_in, 1),   # Span all channels
                padding='valid'
            ),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        """
        x : (B, C, T)
        Returns token embeddings: (B, T, d_model)
        """
        x = x.unsqueeze(1)            # (B, 1, C, T)
        x = self.embed_layer(x)       # (B, d_model*4, C, T)
        x = self.embed_layer2(x)      # (B, d_model, 1, T)
        x = x.squeeze(2)              # (B, d_model, T)
        x = x.permute(0, 2, 1)        # (B, T, d_model)
        return x


# tokenembedding + LSTM + DANN Model
# - Token embedding through convolution
# - LSTM sequence model
# - Task classifier (main task)
# - Domain classifier (subject classifier) with GRL
class DARNet_LSTM_DANN(nn.Module):
    def __init__(self, c_in=32, d_model=16, hidden=64,
                 num_classes=2, num_subjects=30):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension per token.
        hidden : int
            Hidden size of the LSTM.
        num_classes : int
            Number of output task classes.
        num_subjects : int
            Number of domain classes (e.g., subjects).
        """
        super().__init__()

        # Embed raw data into feature tokens
        self.token_embed = TokenEmbedding(c_in, d_model)

        # Bidirectional LSTM for temporal feature extraction
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Task classifier (main AAD task)
        self.classifier = nn.Linear(hidden * 2, num_classes)

        # Domain classifier (subject prediction)
        # Used via gradient reversal for adversarial training
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_subjects)
        )

    def forward_features(self,x):

        x = x.unsqueeze(1)
        # Temporal Module
        temporal = self.token_embed.embed_layer(x)
        # Spatial Module
        spatial = self.token_embed.embed_layer2(temporal)
        # Tokens
        emb = spatial.squeeze(2).permute(0,2,1)
        # LSTMs
        lstm,_ = self.lstm(emb)
        pooled = lstm.mean(dim=1)
        return {
            "temporal":temporal,
            "spatial":spatial,
            "lstm":lstm,
            "pooled":pooled
        }


    def forward(self, x, lambd=0.0):
        """
        Forward pass with optional gradient reversal.

        Parameters
        ----------
        x : (B, C, T)
            Input EEG batch.
        lambd : float
            Lambda for gradient reversal (0 during evaluation).
        """
        # Token embedding
        emb = self.token_embed(x)                          # (B, T, d_model)

        # Sequence modeling
        features, _ = self.lstm(emb)                       # (B, T, hidden*2)

        # 3. Global average pooling across time
        feat = features.mean(dim=1)                        # (B, hidden*2)

        # Task prediction
        logits_task = self.classifier(feat)

        # Domain prediction (with gradient reversal)
        rev = grad_reverse(feat, lambd)                    # Reverse gradients
        logits_domain = self.domain_classifier(rev)

        return logits_task, logits_domain


# **Function to extract the activations from each layer of the model**

In [ ]:
# Extracting the activations from each module of sparse autoencoders
def extract_activations(model, loader, device):
    """
    Function to extract the activations from each layer of the model

    Parameters
    ----------
    model : A model trained on EEG-AAD data
    loader : Dataloader which contains the raw data to pass to trained model.
    device : The device on which the model is loaded

    Returns: A dictionary with activations of each layer
    """

    # Set Model to eval mode
    model.eval()
    # Initialize the lists to accumulate activations, Labels and subject ids
    temporal_all=[]
    spatial_all=[]
    lstm_all=[]
    pooled_all=[]
    labels_all=[]
    subjects_all=[]


    with torch.no_grad():
        # For each sample
        for x,y,s in tqdm(loader):
            # Pass the data to cuda
            x=x.to(device)
            # Retreiving Activations from each layer
            acts=model.forward_features(x)
            # activations from temporal feature extraction module (shape: B, 64, 32, T) 
            # after mean pool shape is (B, 64)
            temporal=acts["temporal"].mean(dim=(2,3))  

            # activations from spatial feature extraction module (shape: B, 16, 1, T) 
            # after mean pool shape is (B, 16)
            spatial=acts["spatial"].mean(dim=(2,3))

            # activations from LSTM module (shape: B, T, 128) 
            # after mean pool shape is (B, 128)
            lstm=acts["lstm"].mean(dim=1)

            # activations from mean pooled module
            pooled=acts["pooled"]

            # accumulating the activations
            temporal_all.append(temporal.cpu())
            spatial_all.append(spatial.cpu())
            lstm_all.append(lstm.cpu())
            pooled_all.append(pooled.cpu())
            labels_all.append(y)
            subjects_all.append(s)


    # Store all the activations in a dictionary
    outputs={
        "temporal":torch.cat(temporal_all),
        "spatial":torch.cat(spatial_all),
        "lstm":torch.cat(lstm_all),
        "pooled":torch.cat(pooled_all),
        "labels":torch.cat(labels_all),
        "subjects":torch.cat(subjects_all)

    }

    return outputs

# **Function to save the folders** 

In [ ]:
def save_outputs(outputs,folder):
    os.makedirs(folder,exist_ok=True)
    for k,v in outputs.items():
        torch.save(
            v,
            os.path.join(folder,f"{k}.pt")
        )

# **Activation retreiving pipeline**

In [ ]:
# Paths to training and validation HDF5 datasets
train_h5_path = '/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/train_data.h5'
val_h5_path   = '/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/val_data.h5'

# Load data from .h5 files (features, labels, subject IDs)
X_train, y_train, s_train = load_h5_dataset(train_h5_path)
X_val,   y_val,   s_val   = load_h5_dataset(val_h5_path)

# Subject ID arrays sometimes have an extra singleton dimension → remove it
s_train = s_train.squeeze()
s_val   = s_val.squeeze()

# Convert subject IDs to 0-based consecutive integer indices
# This ensures subject IDs are compatible with PyTorch operations.
unique_subjects = np.unique(np.concatenate([s_train, s_val]))     # Find all unique subject IDs
subject2idx = {sub: i for i, sub in enumerate(unique_subjects)}   # Map original ID → new index

# Apply conversion to both sets
s_train = np.array([subject2idx[s] for s in s_train])
s_val   = np.array([subject2idx[s] for s in s_val])

# Wrap datasets in PyTorch Dataset objects
# These handle indexing and formatting into PyTorch tensors.
train_dataset = CustomDatasets(X_train, y_train, s_train)
val_dataset   = CustomDatasets(X_val, y_val, s_val)

# (Optional sanity check) Print dataset shapes
print(f"Train data: {X_train.shape}, labels: {y_train.shape}")
print(f"Val data:   {X_val.shape}, labels: {y_val.shape}")

# Build DataLoaders for batching, shuffling, and parallel reading

train_loader = DataLoader(
    train_dataset,
    batch_size=128,   # Number of samples per batch
    shuffle=True,     # Shuffle training samples each epoch
    drop_last=True    # Ensures all batches are full-sized
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,    # No need to shuffle validation data
    drop_last=False   # Keep all samples
)

# Define the model and load the pretrained weights
model = DARNet_LSTM_DANN(d_model = 8, num_subjects=len(unique_subjects))
model.load_state_dict(
torch.load("/kaggle/input/notebooks/sumanpunshi123/eeg-aad-mm-aad-data/best_model.pth", map_location="cuda"))
model=model.to("cuda")

# Extract the activations from a trained model
train_outputs=extract_activations(model, train_loader, "cuda")
val_outputs=extract_activations(model, val_loader, "cuda")

# Save the output dictionaries
save_outputs(train_outputs, "activations/train")
save_outputs(val_outputs, "activations/val")
print("Done.")